In [12]:
from langchain_chroma import Chroma
import os
import chromadb
import pathlib
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

Loading the Documents 

In [16]:
# Source 1 - the PDF. PyPDFLoader returns one document per page.

pdf_docs = PyPDFLoader("Lagos_Education_RAG_Knowledge_Base.pdf").load()


print("Pages loaded from the PDF :", len(pdf_docs))

print()
print("First 300 characters of the PDF:")
print(pdf_docs[0].page_content[:300])


Pages loaded from the PDF : 33

First 300 characters of the PDF:
Page1
LagosLGAEducationResources:RAG
KnowledgeBase
Education Resource-Gap Intelligence Tool · 20 Lagos LGAs · 2016/17–2019/20 and 2024/25
AboutThisDocument
Thisdocumentdescribestheeducationresourcesofpublicandprivateschoolsinthe20
LocalGovernmentAreas(LGAs)ofLagosState,Nigeria.Itistheknowledgebasefo


Splitting the documents into chunks

In [19]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=50
)

# The PDF pages are already documents, so we split the documents
pdf_chunks = splitter.split_documents(pdf_docs)

print("Chunks from the PDF :", len(pdf_chunks))

chosen_chunks = pdf_chunks              


# Chroma only needs the plain text of each chunk
documents = [chunk.page_content for chunk in chosen_chunks]

print()
print("Documents ready to embed:", len(documents))

Chunks from the PDF : 113

Documents ready to embed: 113


In [20]:
#use this to comfirm the number documents to embed 
print(f"Total Documents to be embedded {len(documents)}\n")

Total Documents to be embedded 113



Loading the API keys from your `.env` file.

In [23]:
load_dotenv()

# ---------------------------------------------------------------
# PROVIDER CONFIG
# The API keys are secret, so they live in the .env file.
# Everything else is written out here so you can see it.
#
# NOTE: the chat model and the embedding model run on TWO DIFFERENT
# servers. Each one needs its own base URL and its own key.
# Do not mix them up.
# ---------------------------------------------------------------

# --- Chat model ---
api_key         = os.getenv("OPENAI_API_KEY")
base_url        = "https://gemma-4.publicaai.com/v1"
chat_model_name = "llama-3.3-70b-versatile"

# --- Embedding model (different server, different key) ---
embedding_api_key    = os.getenv("EMBEDDING_API_KEY")
embedding_base_url   = "https://qwen-embed.publicaai.com/v1"
embedding_model_name = "Qwen/Qwen3-Embedding-0.6B"

print("Chat base URL     :", base_url)
print("Chat model        :", chat_model_name)
print("Embed base URL    :", embedding_base_url)
print("Embedding model   :", embedding_model_name)

Chat base URL     : https://gemma-4.publicaai.com/v1
Chat model        : llama-3.3-70b-versatile
Embed base URL    : https://qwen-embed.publicaai.com/v1
Embedding model   : Qwen/Qwen3-Embedding-0.6B


In [21]:
# Setting Up ChromaDB for embeddings Storage
chroma_path = "chroma_store"



**We are using the `embedding_model_name` above** to convert documents into vector embeddings.


In [24]:
# Initialize the embedding model
embeddings = OpenAIEmbeddings(
    model=embedding_model_name,
    api_key=embedding_api_key,
    base_url=embedding_base_url

)

- Creates a **vector database (Chroma)** for the documents.
- Adds the **embedded texts** to the database.

In [25]:
# Initialize ChromaDB
# lerkb = lagos education rag knowledge base
lerkbvdb = Chroma(
    collection_name="lerkb",
    embedding_function=embeddings,
    persist_directory=chroma_path
)


In [26]:
# Add documents to the vector database
lerkbvdb.add_texts(
    texts=documents
)

['4a763ec2-cd29-4276-a22a-f2626148cb8b',
 '54443d0e-69e6-43a3-88e1-3bf81ecd7920',
 '00a4b227-a479-4ffd-ae96-cfe99335081e',
 '71a5247f-b0f9-4715-a43e-13bbc39b95a1',
 '59528837-1a66-4da4-a362-8866c97e5c0f',
 'e0d0efbd-efd1-4a8f-be08-7fd4eaabd671',
 'a1a1f918-aded-4873-989a-1b33e085e192',
 '2e0274bb-34d8-4950-9e3d-d064671e6bc6',
 '0beeffeb-a9f1-4965-a629-70d78c21a414',
 '86408002-f8d5-457e-be9a-5e3b9d590dff',
 '4c8e98d9-c2fc-439c-bd57-a9ff96fd2e99',
 '6ff077da-813f-4a62-8cca-b74ece3e6961',
 '43283d72-e8fd-4299-babf-fd45932bbc57',
 '216642f0-3477-4939-8cde-e54bd8e44cc7',
 'e4ff2c3b-06ae-4f9e-9b1a-d47f38f41764',
 '294c51ed-7cee-460e-960b-784753d254fe',
 'c771777d-ac46-4a56-afd9-99a19be06dcd',
 '974c032a-b046-45f1-9376-10bb363b6181',
 '7faccf12-9344-4959-a7f7-6bf0c0765da4',
 '83c5f229-4b96-4e05-8483-5e15a441abf0',
 '1fb9efbf-3633-4f3c-82a3-d8e05a489ad9',
 'f42037c9-6be8-498c-993a-13fda8507bbe',
 '55920a58-73b3-48b0-945a-0e25690d2b76',
 '16b272f9-6bea-4fd6-9e08-0a05b7181b49',
 'fbf39fff-adff-